In [124]:
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def user_profile_def(user_id, view_history, programs_tfidf):
    view_history = pd.read_csv("data/view_history.csv")
    view_history_user = view_history[view_history["user_id"] == user_id]
    programs_tfidf = pd.read_csv("data/programs_tfidf.csv")

    user_profile = pd.merge(view_history_user, programs_tfidf, on="program_id", how="left")
    user_profile.dropna(axis=0)

    tfidf_cols = user_profile.columns[user_profile.columns.get_loc('save') + 1:]

    user_profile[tfidf_cols] = user_profile[tfidf_cols].multiply(user_profile['listen_ratio'], axis=0)

    user_profile = user_profile.groupby('user_id')[tfidf_cols].mean().reset_index()
    return user_profile

def cosine_similarity_def(user_profile, programs_tfidf):
    user_profile = pd.DataFrame(user_profile)
    tfidf_cols = [col for col in user_profile.columns if col != 'user_id']
    user_vectors = user_profile[tfidf_cols].values
    program_vectors = programs_tfidf[tfidf_cols].values

    sim_matrix = cosine_similarity(user_vectors, program_vectors)
     
    sim_df = pd.DataFrame(
        sim_matrix,
        index=user_profile['user_id'],
        columns=programs_tfidf['program_id']
    )

    return sim_df

def top_50_recommendations(user_id, similarity_df, threshold_amount=50):
    # Adjust this threshold to whatever we need for the exposure fairness and diversity/mmr
    similarity_df = similarity_df.transpose().sort_values(by=user_id, ascending=False)
    similarity_df = pd.merge(similarity_df, pd.read_csv("data/programs.csv"), on="program_id", how="left").dropna()
    return similarity_df.iloc[:threshold_amount]

def exposure_fairness(top_50_recommendations):
    return top_50_recommendations.sort_values(by="inclusion_score", ascending=False).reset_index(drop=True)

def recommendation_content(user_id, view_history, programs_tfidf):
    user_profile = user_profile_def(user_id, view_history, programs_tfidf)
    
    similarity_df = cosine_similarity_def(user_profile, programs_tfidf)

    top50_programs = top_50_recommendations(user_id, similarity_df)

    exposure = exposure_fairness(top50_programs)

    return exposure


In [16]:
os.chdir("/home/anass/university/msc_applied_data_science/INFOMPPM/INFOMPPM")


In [125]:
recommendation_content("U9999", pd.read_csv("data/view_history.csv"), pd.read_csv("data/programs_tfidf.csv"))

,program_id,U9999,category,title,tags,description,image,duration_txt,duration_sec,first_broadcast,synopsis_small,synopsis_medium,synopsis_large,popularity_group,popularity_rank,quality_label,view_count,I_i,inclusion_score
0,3579,0.71697,entertainment,I Can See Your Voice - Series 1: Episode 8,"BBC, iPlayer, TV, I Can See Your Voice, Series...",Paddy McGuinness and the panel are joined by s...,https://ichef.bbci.co.uk/images/ic/1200x675/p0...,48 mins,2876.0,29 May 2021,Paddy McGuinness and the panel are joined by s...,Host Paddy McGuinness is joined by celebrity i...,Host Paddy McGuinness is joined by celebrity i...,tail,2023.0,good,22.0,0.833700,0.833700
1,3606,0.71697,entertainment,The Graham Norton Show - Series 29: Episode 17,"BBC, iPlayer, TV, The Graham Norton Show, Seri...","Courteney Cox, Taron Egerton, Minnie Driver, U...",https://ichef.bbci.co.uk/images/ic/1200x675/p0...,50 mins,2975.0,10:35pm 4 Feb 2022,"Courteney Cox, Taron Egerton, Minnie Driver, U...","Graham is joined by actors Courteney Cox, Taro...",Graham's guests are Friends star Courteney Cox...,tail,2050.0,good,19.0,0.761075,0.761075
2,867,0.71697,music,Radio 1's Live Lounge - Mahalia,"BBC, iPlayer, TV, Radio 1s Live Lounge, Mahalia",Mahalia performs her single Simmer and covers ...,https://ichef.bbci.co.uk/images/ic/1200x675/p0...,11 mins,642.0,12 Sep 2019,Mahalia performs her single 'Simmer' and cover...,Mahalia performs her single 'Simmer' in the Li...,No data found,mid,131.0,good,98.0,0.706796,0.706796
3,2074,0.71697,entertainment,Canada's Drag Race - Series 2: 1. Lost and Fierce,"BBC, iPlayer, TV, Canadas Drag Race, Series 2:...",The Queen’s show off their design skills in a ...,https://ichef.bbci.co.uk/images/ic/1200x675/p0...,60 mins,3575.0,17 Dec 2021,The Queen’s show off their design skills in a ...,Twelve of Canada’s top Drag Queens show off th...,Canada's Drag Race is a 10 x 1 hour competitio...,mid,518.0,good,130.0,0.699808,0.699808
4,4437,0.71697,documentaries,Couples Therapy - Series 1: Episode 6,"BBC, iPlayer, TV, Couples Therapy, Series 1: E...",Elaine and DeSean argue about the impact of ra...,https://ichef.bbci.co.uk/images/ic/1200x675/p0...,24 mins,1465.0,10 Jan 2022,Elaine and DeSean argue about the impact of ra...,The couples continue their weekly therapy with...,No data found,mid,293.0,good,265.0,0.699401,0.699401
5,1901,0.71697,entertainment,Top Gear - Best Of: Episode 5,"BBC, iPlayer, TV, Top Gear, Best Of: Episode 5",Matt takes Chris on a hunt for Bigfoot and Ror...,https://ichef.bbci.co.uk/images/ic/1200x675/p0...,59 mins,3553.0,7pm 13 Jan 2019,Matt takes Chris on a hunt for Bigfoot and Ror...,Featuring the greatest Top Gear adventures fro...,Featuring the greatest Top Gear adventures fro...,mid,345.0,good,128.0,0.696634,0.696634
6,2056,0.71697,entertainment,Bargain Hunt - Series 61: Ardingly 1,"BBC, iPlayer, TV, Bargain Hunt, Series 61: Ar...",Eric Knowles is in Sussex with experts Raj Bis...,https://ichef.bbci.co.uk/images/ic/1200x675/p0...,44 mins,2610.0,12:15pm 7 Mar 2022,Eric Knowles is in Sussex with experts Raj Bis...,"Eric Knowles visits Ardingly in Sussex, where ...","Eric Knowles visits Ardingly in Sussex, where ...",mid,500.0,good,152.0,0.690080,0.690080
7,2042,0.71697,entertainment,Ryan Tricks On The Streets - Series 1: 3. Help...,"BBC, iPlayer, TV, Ryan Tricks On The Streets, ...","Meeting a group of young people, Ryan encourag...",https://ichef.bbci.co.uk/images/ic/1200x675/p0...,7 mins,420.0,4 Nov 2018,"Meeting a group of young people, Ryan encourag...",Ryan gives back to a group of young people who...,When Ryan was a kid he almost went off the rai...,mid,486.0,good,116.0,0.689234,0.689234
8,3109,0.71697,entertainment,The Hit List - Series 3: Episode 9,"BBC, iPlayer, TV, The Hit List, Series 3: Epis...","Featuring teams from Glasgow, London and Belfast.",https://ichef.bbci.co.uk/images/ic/1200x675/p0...,44 mins,2636.0,6:40pm 24 Apr 2021,"Featuring teams from Glasgow, London and Belfast.",Father and daughter Bill and 